# GF180MCU-D RF PA Auto-Generation Setup (macOS)

This notebook is the RFPA version of the original `LLM_Amplifier_Sizing` setup flow. It uses this checkout directly instead of cloning the reference repository, prepares the GF180MCU-D CircuitCollector backend, builds starter gm/ID LUT assets from `Sizing/backup`, starts the API, and runs quick RFPA seed simulations.

The goal of this notebook is initial results, not final PA signoff. The current backend gives schematic seed evidence for topology ranking and iteration; layout, EM, package/board reference planes, full load-pull, and final matching closure remain future work.


## 1. Locate This RFPA Checkout

Run this notebook from the RFPA repository root, or set `RFPA_ROOT` before starting Jupyter.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import urllib.request


def find_repo_root(start: Path) -> Path:
    env_root = os.environ.get("RFPA_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "AnalogAgent" / "skills" / "rf-power-amplifier").is_dir():
            return root
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "AnalogAgent" / "skills" / "rf-power-amplifier").is_dir()
            and (candidate / "CircuitCollector" / "setup.py").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Could not find RFPA repo root. Start Jupyter from the repo root or set RFPA_ROOT."
    )


REPO_ROOT = find_repo_root(Path.cwd())
ANALOG_AGENT_ROOT = REPO_ROOT / "AnalogAgent"
SKILL_ROOT = ANALOG_AGENT_ROOT / "skills" / "rf-power-amplifier"
CC_REPO = REPO_ROOT / "CircuitCollector"
CC_ROOT = CC_REPO / "CircuitCollector"

print("repo root:", REPO_ROOT)
print("AnalogAgent:", ANALOG_AGENT_ROOT)
print("CircuitCollector repo:", CC_REPO)
print("CircuitCollector package root:", CC_ROOT)
print("RFPA skill root:", SKILL_ROOT)


## 2. Install Python Dependencies

This installs CircuitCollector from the local folder in editable mode, plus the small analysis dependencies used by the LUT builder and smoke-result table.


In [ ]:
install_cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-e",
    str(CC_REPO),
    "pandas>=1.5",
    "requests>=2.31",
    "anyio>=4.8,<5",
]
print(" ".join(install_cmd))
subprocess.check_call(install_cmd)


## 3. Check ngspice

On Apple Silicon, conda-forge does not provide a native `osx-arm64` ngspice package. This cell uses an existing `ngspice` if present, otherwise it tries Homebrew on macOS.


In [ ]:
def prepend_path(path: Path) -> None:
    path = path.expanduser()
    if path.is_dir():
        os.environ["PATH"] = str(path) + os.pathsep + os.environ.get("PATH", "")


for candidate in [Path("/opt/homebrew/bin"), Path("/usr/local/bin")]:
    prepend_path(candidate)

if shutil.which("ngspice") is None:
    brew = shutil.which("brew")
    if sys.platform == "darwin" and brew:
        subprocess.check_call([brew, "install", "ngspice"])
    else:
        raise RuntimeError("ngspice not found. Install ngspice, then rerun this cell.")

ngspice = shutil.which("ngspice")
print("ngspice:", ngspice)
print(subprocess.check_output([ngspice, "-v"], text=True).splitlines()[0])


## 4. Link GF180MCU-D PDK

The PDK is not committed into this repo. This cell links a local GF180MCU-D install into the path expected by the TOML configs: `CircuitCollector/CircuitCollector/PDK/gf180mcuD`.

If auto-detection fails, set `GF180MCU_D_PDK` to the directory that contains `libs.tech/ngspice/design.ngspice`.


In [ ]:
PDK_DEST = CC_ROOT / "PDK" / "gf180mcuD"
REQUIRED_PDK_FILES = [
    Path("libs.tech/ngspice/design.ngspice"),
    Path("libs.tech/ngspice/sm141064.ngspice"),
]


def pdk_is_valid(path: Path) -> bool:
    return path is not None and all((path / rel).is_file() for rel in REQUIRED_PDK_FILES)


def pdk_candidates() -> list[Path]:
    candidates: list[Path] = []
    for env_name in ["GF180MCU_D_PDK", "GF180_PDK", "PDK_ROOT"]:
        value = os.environ.get(env_name)
        if value:
            path = Path(value).expanduser()
            candidates.extend([path, path / "gf180mcuD"])

    version_roots = [
        Path("/foss/pdks/ciel/gf180mcu/versions"),
        Path("/foss/pdks/volare/gf180mcu/versions"),
        Path.home() / "Desktop/foss/pdks/volare/gf180mcu/versions",
    ]
    for root in version_roots:
        if root.is_dir():
            candidates.extend(sorted(root.glob("*/gf180mcuD"), reverse=True))

    candidates.extend(
        [
            Path.home()
            / "eda/designs/Book-on-gm-ID-design/starter_files_open_source_tools/gf180mcuD",
            Path.home()
            / "Desktop/foss/pdks/volare/gf180mcu/versions/0fe599b2afb6708d281543108caf8310912f54af/gf180mcuD",
        ]
    )
    return candidates


if pdk_is_valid(PDK_DEST):
    print("PDK already valid:", PDK_DEST.resolve())
else:
    selected = next((path for path in pdk_candidates() if pdk_is_valid(path)), None)
    if selected is None:
        checked = "\n".join(str(path) for path in pdk_candidates())
        raise FileNotFoundError(
            "Could not auto-detect GF180MCU-D PDK. Set GF180MCU_D_PDK and rerun.\n"
            f"Checked:\n{checked}"
        )

    PDK_DEST.parent.mkdir(parents=True, exist_ok=True)
    if PDK_DEST.is_symlink() or PDK_DEST.exists():
        if PDK_DEST.is_symlink():
            PDK_DEST.unlink()
        else:
            raise RuntimeError(f"PDK destination exists but is not valid/symlink: {PDK_DEST}")
    os.symlink(selected.resolve(), PDK_DEST, target_is_directory=True)
    print("Linked PDK:", PDK_DEST, "->", selected.resolve())


## 5. Build Starter GF180 gm/ID LUT Assets

The reference amplifier-sizing project has processed LUT assets. This RFPA repo had the raw GF180 room-temperature typical sweeps in `Sizing/backup`; this cell converts them into the `AnalogAgent/asset_gf180mcuD` layout used by `lut_lookup.py`.


In [ ]:
lut_builder = ANALOG_AGENT_ROOT / "scripts" / "build_gf180_lut_assets.py"
subprocess.check_call([sys.executable, str(lut_builder)])

sys.path.insert(0, str(ANALOG_AGENT_ROOT))
from scripts.lut_lookup import list_available_L, lut_query

print("nfet lengths (um):", list_available_L("nfet"))
print("pfet lengths (um):", list_available_L("pfet"))
print("nfet id_w at gm/ID=10, L=0.28um:", lut_query("nfet", "id_w", 0.28, gm_id_val=10.0))
print("pfet id_w at gm/ID=10, L=0.28um:", lut_query("pfet", "id_w", 0.28, gm_id_val=10.0))


## 6. Start CircuitCollector API

This starts a local FastAPI server on port `8001` by default. Set `RFPA_API_PORT` before launching Jupyter to use a different port.


In [ ]:
API_HOST = "127.0.0.1"
API_PORT = int(os.environ.get("RFPA_API_PORT", "8001"))
API_URL = f"http://{API_HOST}:{API_PORT}"
API_LOG = REPO_ROOT / "circuitcollector_api.log"

# Stop stale servers from earlier notebook runs.
try:
    pgrep = subprocess.run(
        ["pgrep", "-f", "CircuitCollector.api.main:app"],
        capture_output=True,
        text=True,
        check=False,
    )
    for pid_text in pgrep.stdout.splitlines():
        pid = int(pid_text.strip())
        if pid != os.getpid():
            try:
                os.kill(pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
    time.sleep(1.0)
except FileNotFoundError:
    pass

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    [str(CC_REPO), str(REPO_ROOT), env.get("PYTHONPATH", "")]
)

log_file = API_LOG.open("w")
process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "CircuitCollector.api.main:app",
        "--host",
        API_HOST,
        "--port",
        str(API_PORT),
    ],
    cwd=str(CC_REPO),
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("started pid:", process.pid)
print("API URL:", API_URL)
print("log:", API_LOG)


## 7. Health Check


In [ ]:
last_error = None
for attempt in range(60):
    try:
        with urllib.request.urlopen(API_URL + "/health", timeout=2) as response:
            print("API healthy:", response.read().decode())
            break
    except Exception as exc:
        last_error = exc
        time.sleep(1)
else:
    tail = API_LOG.read_text(errors="replace").splitlines()[-80:] if API_LOG.exists() else []
    raise RuntimeError(
        f"API did not become healthy: {last_error}\n"
        + "\n".join(tail)
    )


## 8. RFPA Backend Coverage

This checks that each RFPA topology has both a TOML config and a netlist template.


In [ ]:
coverage_cmd = [
    sys.executable,
    str(SKILL_ROOT / "scripts" / "check_backend_coverage.py"),
    "--backend-root",
    str(CC_ROOT),
]
coverage = subprocess.run(coverage_cmd, capture_output=True, text=True, check=True)
print(coverage.stdout)


## 9. Quick Initial Results

The default quick run compares the closest current seeds:

- `class_ab_single_ended`: closest to the 2 mW target, but current small-signal stability/matching estimates are not yet clean.
- `two_stage_single_ended`: more stable in the rough two-port estimate, but currently under the 2 mW target.

Edit `TOPOLOGIES` to include any other seeded topology.


In [ ]:
TOPOLOGIES = ["class_ab_single_ended", "two_stage_single_ended"]

smoke_cmd = [
    sys.executable,
    str(SKILL_ROOT / "scripts" / "smoke_test_backend.py"),
    "--backend-root",
    str(CC_ROOT),
    "--api-url",
    API_URL + "/simulate/",
    "--quick",
]
for topology in TOPOLOGIES:
    smoke_cmd.extend(["--topology", topology])

smoke = subprocess.run(smoke_cmd, capture_output=True, text=True, check=True)
print(smoke.stdout)

json_start = smoke.stdout.find('{\n  "backend_root"')
if json_start == -1:
    raise RuntimeError("Could not find JSON result in smoke-test output")
smoke_data = json.loads(smoke.stdout[json_start:])

import pandas as pd
rows = []
for topology, result in smoke_data["results"].items():
    metrics = result.get("parseable_metrics", {})
    rows.append(
        {
            "topology": topology,
            "status": result.get("status"),
            "pout_mw": None if metrics.get("pout_w") is None else 1e3 * metrics.get("pout_w"),
            "pout_dbm": metrics.get("pout_dbm"),
            "pdc_mw": None if metrics.get("pdc_w") is None else 1e3 * metrics.get("pdc_w"),
            "gain_db": metrics.get("gain_db"),
            "pae_percent": metrics.get("pae"),
            "drain_eff_percent": metrics.get("drain_efficiency"),
            "iout_pk_ma": None if metrics.get("iout_pk_est") is None else 1e3 * metrics.get("iout_pk_est"),
            "h2_dbc": metrics.get("h2_dbc"),
            "h3_dbc": metrics.get("h3_dbc"),
            "s11_db_est": metrics.get("s11_db"),
            "s22_db_est": metrics.get("s22_db"),
            "k_est": metrics.get("stability_k"),
            "mu_est": metrics.get("stability_mu"),
            "idc_pass": metrics.get("idc_limit_pass"),
            "iout_pk_pass": metrics.get("iout_pk_limit_pass"),
        }
    )

summary = pd.DataFrame(rows)
display(summary)


## 10. How This Compares To `LLM_Amplifier_Sizing`

The reference project is a converged OTA/op-amp sizing loop: processed gm/ID LUTs, procedural skill files, CircuitCollector/ngspice verification, root-cause iteration, and optional numerical optimization.

This RFPA project now has the same basic skeleton for initial results: RFPA skill files, GF180 backend configs/netlists, starter gm/ID LUT assets, an API path, and smoke-tested PA seed metrics. The current gap is the automatic design loop: topology ranking exists as documentation and seed configs, but there is not yet a mature optimizer/load-pull/matching loop equivalent to the OTA flow.

Practical status after this notebook: you can get initial simulated RFPA seed results now. To turn this into full auto-generation, the next engineering step is to automate parameter updates around the two strongest seeds: stabilize/match `class_ab_single_ended`, or raise `two_stage_single_ended` output power while preserving its better stability estimate.


## 11. Suggested Agent Prompt After Setup

Once the smoke test above runs, use this prompt from the repo root:

```text
Use the RF power amplifier skill to iterate a GF180MCU-D on-chip-core RF PA for 250 MHz, VDD=3.3 V, 50 ohm load, target Pout=2 mW, max DC current=10 mA, max output peak current=10 mA.

CircuitCollector API: http://127.0.0.1:8001/simulate/
Start from class_ab_single_ended and two_stage_single_ended. Treat S11/S22 from the current testbench as rough estimates, not signoff. Use the generated GF180 gm/ID LUT assets only as typical/25C first-pass sizing guidance.
```
